# VWAP Leak-Free Reranking Notebook


In [1]:
from pathlib import Path
import json
import sys
from IPython.display import Image, Markdown, display
import pandas as pd

root = Path.cwd().resolve()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
if not (root / "pyproject.toml").exists():
    raise RuntimeError("Could not locate repository root.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 120)


In [2]:
OUTPUT_DIR = Path(r"D:\Business\Trading\VSCODE\algo-trading-intraday-research\data\exports\vwap_reranking_final_20260324")
print(OUTPUT_DIR)


D:\Business\Trading\VSCODE\algo-trading-intraday-research\data\exports\vwap_reranking_final_20260324


## 1) Paper Baseline Reference


In [3]:
display(Markdown((OUTPUT_DIR / "baseline" / "paper_baseline_reference.md").read_text(encoding="utf-8")))


# Paper Baseline Reference

- Official reference baseline is rerun under the current leak-free semantics.
- Signal: previous close vs previous session VWAP.
- Execution: next open, always-in-market during RTH, flat overnight.
- Session: `09:30:00` -> `16:00:00`.
- Costs/profile: `paper_reference`.

```text
        strategy_id                     signal_rule      execution_rule              session  flat_overnight       quantity_mode  initial_capital_usd execution_profile  overall_total_trades  overall_net_pnl  overall_profit_factor  overall_sharpe_ratio  overall_max_drawdown  overall_worst_daily_loss_usd  overall_daily_loss_limit_breach_freq  overall_trailing_drawdown_breach_freq  oos_total_trades  oos_net_pnl  oos_profit_factor  oos_sharpe_ratio  oos_max_drawdown  oos_worst_daily_loss_usd  oos_daily_loss_limit_breach_freq  oos_trailing_drawdown_breach_freq  oos_net_pnl_slippage_x2  oos_profit_factor_slippage_x2  oos_sharpe_ratio_slippage_x2
paper_vwap_baseline close[t-1] vs session VWAP[t-1] next open leak-free 09:30:00 -> 16:00:00            True paper_full_notional              25000.0   paper_reference                 20729         -25008.0                0.92629             -0.899476              -25373.5                       -1380.0                              0.002862                               0.988552               613      -2376.5           0.766288         -0.706673           -2518.5                   -1380.0                          0.001905                           0.940952                  -2989.5                       0.720529                     -0.860157
```


## 2) Full Variant Comparison


In [4]:
display(pd.read_csv(OUTPUT_DIR / "catalog" / "variant_catalog.csv"))
display(pd.read_csv(OUTPUT_DIR / "reranking_summary.csv"))


,display_order,strategy_id,role,family,mode,execution_profile,what_changes_vs_baseline,time_windows,slope_lookback,slope_threshold,require_vwap_slope_alignment,max_vwap_distance_atr,atr_buffer,stop_buffer,max_trades_per_day,max_losses_per_day,daily_stop_threshold_usd,consecutive_losses_threshold,exit_on_vwap_recross,notes
0,1,paper_vwap_baseline,paper_baseline_reference,baseline,target_position,paper_reference,"Reference officielle: close[t-1] vs VWAP[t-1],...",full_rth,5,0.0,False,NaN,0.25,0.25,NaN,NaN,NaN,NaN,True,Exact paper logic: previous close relative to ...
1,2,baseline_futures_adapted,realistic_baseline_reference,baseline,target_position,repo_realistic,"Meme signal que la baseline paper, mais sous s...",full_rth,5,0.0,False,NaN,0.25,0.25,NaN,NaN,NaN,NaN,True,"Same signal logic as the paper baseline, but p..."
2,3,vwap_time_filtered_baseline,candidate,prop_variant,target_position,repo_realistic,Baseline paper gatee par une fenetre horaire s...,09:35:00->11:30:00|15:00:00->15:50:00,5,0.0,False,NaN,0.25,0.25,NaN,NaN,NaN,NaN,True,Paper signal with allowed entries and reversal...
3,4,vwap_baseline_trade_capped,candidate,prop_variant,target_position,repo_realistic,Baseline paper avec plafond dur sur le nombre ...,full_rth,5,0.0,False,NaN,0.25,0.25,6.0,NaN,NaN,NaN,True,Paper signal with a hard cap on the number of ...
4,5,vwap_baseline_regime_filtered,candidate,prop_variant,target_position,repo_realistic,Baseline paper gatee par un filtre de pente VW...,full_rth,5,0.0,True,1.0,0.25,0.25,8.0,NaN,NaN,NaN,True,Paper signal gated by previous-bar VWAP slope ...
5,6,vwap_baseline_with_killswitch,candidate,prop_variant,target_position,repo_realistic,Baseline paper avec kill switches journaliers ...,full_rth,5,0.0,False,NaN,0.25,0.25,12.0,3.0,750.0,NaN,True,Paper signal with daily loss and trade-count c...
6,7,vwap_reclaim,candidate,prop_variant,discrete,repo_realistic,Variante discrete reclaim simple avec pente VW...,full_rth,5,0.0,False,NaN,0.25,0.25,3.0,NaN,NaN,NaN,True,"VWAP reclaim after compression, confirmed by p..."
7,8,vwap_reclaim_with_prop_overlay,candidate,prop_variant,discrete,repo_realistic,Variante reclaim simple avec overlay prop: fen...,09:35:00->11:30:00|15:00:00->15:50:00,5,0.0,False,NaN,0.25,0.25,2.0,2.0,500.0,2.0,True,"Reclaim variant with time filter, hard daily s..."


,strategy_id,role,family,mode,execution_profile,what_changes_vs_baseline,overall_net_pnl,oos_net_pnl,oos_profit_factor,oos_sharpe_ratio,oos_max_drawdown,oos_expectancy_per_trade,oos_total_trades,pnl_nominal,pnl_slip_x2,pf_nominal,pf_slip_x2,sharpe_nominal,sharpe_slip_x2,dd_nominal,dd_slip_x2,delta_pnl_nominal_vs_slip_x2,pass_fail_cost_stress,positive_oos_splits,total_splits,mean_oos_net_pnl_splits,mean_oos_profit_factor_splits,mean_oos_sharpe_ratio_splits,worst_oos_split_net_pnl,best_oos_split_net_pnl,pass_fail_splits,worst_daily_loss_usd,daily_loss_limit_breach_freq,trailing_drawdown_breach_freq,avg_trades_per_day,max_trades_per_day,challenge_success_rate_standard,prop_verdict,top_3_day_contribution_pct,top_5_day_contribution_pct,pnl_excluding_top_3_days,pnl_excluding_top_5_days,concentration_verdict,pass_oos_nominal,pass_cost_stress,pass_splits,pass_prop,survives_primary_filter,reranking_score,final_bucket,elimination_reason,rank_within_survivors
0,paper_vwap_baseline,paper_baseline_reference,baseline,target_position,paper_reference,"Reference officielle: close[t-1] vs VWAP[t-1],...",-25008.000000,-2376.500000,0.766288,-0.706673,-2518.500000,-3.876835,613,-2376.500000,-2989.500000,0.766288,0.720529,-0.706673,-0.860157,-2518.500000,-2991.000000,-613.0,False,0,4,-5460.125000,0.585349,-0.950730,-12137.500000,0.000000,False,-1380.000000,0.001905,0.940952,1.167619,53,0.0,non defendable,-0.731959,-0.937092,-4116.000000,-4603.500000,forte dependance aux meilleurs jours,False,False,False,False,False,0,reference_officielle,"Ancre officielle de comparaison, non candidate...",NaN
1,baseline_futures_adapted,realistic_baseline_reference,baseline,target_position,repo_realistic,"Meme signal que la baseline paper, mais sous s...",-42674.500000,-20043.000000,0.900446,-1.412155,-20185.000000,-2.324096,8624,-20043.000000,-28667.000000,0.900446,0.862572,-1.412155,-1.986944,-20185.000000,-28668.500000,-8624.0,False,0,4,-22906.875000,0.893610,-1.521253,-29804.000000,-16787.500000,False,-1931.500000,0.011429,0.872381,16.426667,53,0.0,non defendable,-0.276580,-0.412039,-25586.500000,-28301.500000,forte dependance aux meilleurs jours,False,False,False,False,False,0,baseline_realiste_de_reference,Baseline economique de reference pour juger si...,NaN
2,vwap_baseline_regime_filtered,candidate,prop_variant,target_position,repo_realistic,Baseline paper gatee par un filtre de pente VW...,-49215.500000,-9733.000000,0.873185,-2.177875,-11312.000000,-2.472815,3936,-9733.000000,-13669.000000,0.873185,0.825778,-2.177875,-3.054785,-11312.000000,-14697.000000,-3936.0,False,0,4,-10940.750000,0.867441,-2.293338,-15032.000000,-6688.500000,False,-558.500000,0.000000,0.874286,7.497143,8,0.0,non defendable,-0.136957,-0.211600,-11066.000000,-11792.500000,forte dependance aux meilleurs jours,False,False,False,False,False,0,eliminee immediatement,Ne survit pas au filtre nominal OOS ou au stre...,NaN
3,vwap_baseline_trade_capped,candidate,prop_variant,target_position,repo_realistic,Baseline paper avec plafond dur sur le nombre ...,-9892.000000,-1821.500000,0.976590,-0.225655,-5659.000000,-0.607572,2998,-1821.500000,-4819.500000,0.976590,0.939864,-0.225655,-0.595779,-5659.000000,-6888.000000,-2998.0,False,0,4,-3182.875000,0.963295,-0.352879,-5647.000000,-1119.500000,False,-763.000000,0.000000,0.287619,5.710476,6,0.0,non defendable,-1.624760,-2.602251,-4781.000000,-6561.500000,forte dependance aux meilleurs jours,False,False,False,False,False,0,eliminee immediatement,Ne survit pas au filtre nominal OOS ou au stre...,NaN
4,vwap_baseline_with_killswitch,candidate,prop_variant,target_position,repo_realistic,Baseline paper avec kill switches journaliers ...,-2032.000000,-1060.500000,0.978823,-0.171220,-3451.000000,-0.538052,1971,-1060.500000,-3031.500000,0.978823,0.941227,-0.171220,-0.489675,-3451.000000,-3945.500000,-1971.0,False,1,4,-818.750000,0.987013,-0.103235,-1951.000000,1277.000000,False,-489.500000,0.000000,0.396190,3.754286,7,0.0,non defendable,-2.616690,-4.174446,-38

## 3) Stress x2


In [5]:
display(pd.read_csv(OUTPUT_DIR / "stress_test_summary.csv"))


,strategy_id,role,pnl_nominal,pnl_slip_x2,pf_nominal,pf_slip_x2,sharpe_nominal,sharpe_slip_x2,dd_nominal,dd_slip_x2,delta_pnl_nominal_vs_slip_x2,pass_fail_cost_stress
0,paper_vwap_baseline,paper_baseline_reference,-2376.500000,-2989.500000,0.766288,0.720529,-0.706673,-0.860157,-2518.500000,-2991.000000,-613.0,False
1,baseline_futures_adapted,realistic_baseline_reference,-20043.000000,-28667.000000,0.900446,0.862572,-1.412155,-1.986944,-20185.000000,-28668.500000,-8624.0,False
2,vwap_baseline_regime_filtered,candidate,-9733.000000,-13669.000000,0.873185,0.825778,-2.177875,-3.054785,-11312.000000,-14697.000000,-3936.0,False
3,vwap_baseline_trade_capped,candidate,-1821.500000,-4819.500000,0.976590,0.939864,-0.225655,-0.595779,-5659.000000,-6888.000000,-2998.0,False
4,vwap_baseline_with_killswitch,candidate,-1060.500000,-3031.500000,0.978823,0.941227,-0.171220,-0.489675,-3451.000000,-3945.500000,-1971.0,False
5,vwap_reclaim,candidate,151.053571,132.053571,1.140522,1.121267,0.125315,0.109574,-750.446429,-764.446429,-19.0,True
6,vwap_reclaim_with_prop_overlay,candidate,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,False
7,vwap_time_filtered_baseline,candidate,-11177.500000,-16452.500000,0.929453,0.898919,-0.807055,-1.177932,-14050.500000,-17968.000000,-5275.0,False


## 4) Multi-Split


In [6]:
display(pd.read_csv(OUTPUT_DIR / "split_summary.csv"))


,strategy_id,positive_oos_splits,total_splits,mean_oos_net_pnl,mean_oos_profit_factor,mean_oos_sharpe_ratio,worst_oos_split_net_pnl,best_oos_split_net_pnl,pass_fail_splits
0,paper_vwap_baseline,0,4,-5460.125000,0.585349,-0.950730,-12137.500000,0.000000,False
1,baseline_futures_adapted,0,4,-22906.875000,0.893610,-1.521253,-29804.000000,-16787.500000,False
2,vwap_time_filtered_baseline,0,4,-14417.000000,0.915547,-0.975730,-19417.500000,-10044.000000,False
3,vwap_baseline_trade_capped,0,4,-3182.875000,0.963295,-0.352879,-5647.000000,-1119.500000,False
4,vwap_baseline_regime_filtered,0,4,-10940.750000,0.867441,-2.293338,-15032.000000,-6688.500000,False
5,vwap_baseline_with_killswitch,1,4,-818.750000,0.987013,-0.103235,-1951.000000,1277.000000,False
6,vwap_reclaim,1,4,-38.696429,0.973630,-0.029248,-117.946429,151.053571,False
7,vwap_reclaim_with_prop_overlay,0,4,0.000000,0.000000,0.000000,0.000000,0.000000,False


## 5) Prop Metrics


In [7]:
display(pd.read_csv(OUTPUT_DIR / "prop_summary.csv"))


,strategy_id,role,worst_daily_loss_usd,red_days_freq,days_below_neg_0p25r_freq,days_below_neg_0p5r_freq,days_below_neg_1r_freq,worst_losing_trades_streak,worst_losing_days_streak,max_drawdown,daily_loss_limit_breach_freq,trailing_drawdown_breach_freq,avg_trades_per_day,max_trades_per_day,challenge_success_rate_standard,challenge_bust_rate_standard,prop_verdict
0,paper_vwap_baseline,paper_baseline_reference,-1380.000000,0.969524,NaN,NaN,NaN,51,4,-2518.500000,0.001905,0.940952,1.167619,53,0.0,0.0,non defendable
1,baseline_futures_adapted,realistic_baseline_reference,-1931.500000,0.539048,NaN,NaN,NaN,52,8,-20185.000000,0.011429,0.872381,16.426667,53,0.0,0.0,non defendable
2,vwap_baseline_regime_filtered,candidate,-558.500000,0.560000,NaN,NaN,NaN,10,12,-11312.000000,0.000000,0.874286,7.497143,8,0.0,0.0,non defendable
3,vwap_baseline_trade_capped,candidate,-763.000000,0.641905,NaN,NaN,NaN,24,12,-5659.000000,0.000000,0.287619,5.710476,6,0.0,0.0,non defendable
4,vwap_baseline_with_killswitch,candidate,-489.500000,0.714286,NaN,NaN,NaN,19,14,-3451.000000,0.000000,0.396190,3.754286,7,0.0,0.0,non defendable
5,vwap_reclaim,candidate,-161.892857,0.990476,0.022857,0.015238,0.011429,5,1,-750.446429,0.000000,0.000000,0.036190,1,0.0,0.0,potentiellement compatible sous contraintes pr...
6,vwap_reclaim_with_prop_overlay,candidate,0.000000,1.000000,NaN,NaN,NaN,0,0,0.000000,0.000000,0.000000,0.000000,0,0.0,0.0,non defendable
7,vwap_time_filtered_baseline,candidate,-1578.500000,0.516190,NaN,NaN,NaN,42,12,-14050.500000,0.011429,0.817143,10.047619,32,0.0,0.0,non defendable


## 6) Concentration


In [8]:
display(pd.read_csv(OUTPUT_DIR / "concentration_summary.csv"))


,strategy_id,role,top_3_day_contribution_pct,top_5_day_contribution_pct,pnl_excluding_top_3_days,pnl_excluding_top_5_days,pnl_excluding_best_month,concentration_verdict
0,paper_vwap_baseline,paper_baseline_reference,-0.731959,-0.937092,-4116.000000,-4603.500000,-2376.500000,forte dependance aux meilleurs jours
1,baseline_futures_adapted,realistic_baseline_reference,-0.276580,-0.412039,-25586.500000,-28301.500000,-22736.000000,forte dependance aux meilleurs jours
2,vwap_baseline_regime_filtered,candidate,-0.136957,-0.211600,-11066.000000,-11792.500000,-11520.000000,forte dependance aux meilleurs jours
3,vwap_baseline_trade_capped,candidate,-1.624760,-2.602251,-4781.000000,-6561.500000,-3816.500000,forte dependance aux meilleurs jours
4,vwap_baseline_with_killswitch,candidate,-2.616690,-4.174446,-3835.500000,-5487.500000,-2139.500000,forte dependance aux meilleurs jours
5,vwap_reclaim,candidate,7.527131,8.116326,-985.946429,-1074.946429,-529.446429,forte dependance aux meilleurs jours
6,vwap_reclaim_with_prop_overlay,candidate,0.000000,0.000000,0.000000,0.000000,0.000000,dependance moderee aux outliers
7,vwap_time_filtered_baseline,candidate,-0.528696,-0.756878,-17087.000000,-19637.500000,-14540.500000,forte dependance aux meilleurs jours


## 7) Survivors Heatmaps


In [9]:
path = OUTPUT_DIR / "survivors_heatmap_readouts.csv"
if path.exists():
    display(pd.read_csv(path))
for png in sorted((OUTPUT_DIR / "heatmaps").glob("*.png")):
    display(Image(filename=str(png)))


,verdict,stable_neighbor_share,reference_sharpe_rank_pct,comment,strategy_id,heatmap_name,chart_path


## 8) Final Verdict


In [10]:
verdict = json.loads((OUTPUT_DIR / "final_verdict.json").read_text(encoding="utf-8"))
verdict
display(Markdown((OUTPUT_DIR / "reranking_summary.md").read_text(encoding="utf-8")))


# VWAP Reranking Summary

- Global verdict: `Aucune variante n'est assez robuste pour meriter une validation approfondie supplementaire.`
- Survivors after primary filter: `none`
- Eliminated immediately: `vwap_baseline_regime_filtered, vwap_baseline_trade_capped, vwap_baseline_with_killswitch, vwap_reclaim_with_prop_overlay, vwap_time_filtered_baseline`

```text
                   strategy_id                         role   oos_net_pnl  oos_profit_factor  oos_sharpe_ratio   pnl_slip_x2  positive_oos_splits  total_splits                                          prop_verdict                concentration_verdict                   final_bucket
           paper_vwap_baseline     paper_baseline_reference  -2376.500000           0.766288         -0.706673  -2989.500000                    0             4                                        non defendable forte dependance aux meilleurs jours           reference_officielle
      baseline_futures_adapted realistic_baseline_reference -20043.000000           0.900446         -1.412155 -28667.000000                    0             4                                        non defendable forte dependance aux meilleurs jours baseline_realiste_de_reference
 vwap_baseline_regime_filtered                    candidate  -9733.000000           0.873185         -2.177875 -13669.000000                    0             4                                        non defendable forte dependance aux meilleurs jours         eliminee immediatement
    vwap_baseline_trade_capped                    candidate  -1821.500000           0.976590         -0.225655  -4819.500000                    0             4                                        non defendable forte dependance aux meilleurs jours         eliminee immediatement
 vwap_baseline_with_killswitch                    candidate  -1060.500000           0.978823         -0.171220  -3031.500000                    1             4                                        non defendable forte dependance aux meilleurs jours         eliminee immediatement
                  vwap_reclaim                    candidate    151.053571           1.140522          0.125315    132.053571                    1             4 potentiellement compatible sous contraintes prudentes forte dependance aux meilleurs jours interessante mais trop fragile
vwap_reclaim_with_prop_overlay                    candidate      0.000000           0.000000          0.000000      0.000000                    0             4                                        non defendable      dependance moderee aux outliers         eliminee immediatement
   vwap_time_filtered_baseline                    candidate -11177.500000           0.929453         -0.807055 -16452.500000                    0             4                                        non defendable forte dependance aux meilleurs jours         eliminee immediatement
```
